# Task B-4: Model Versioning and Experimentation

## 1. Objective
To fulfill Task B-4, we compare three different versions of the Prometheus model to determine the optimal configuration for gym occupancy monitoring. We track hyperparameters, architectures, and performance metrics using **MLflow**.

### Models to Compare:
1. **Baseline_v1**: YOLOv10s (Small), imgsz=640, 50 epochs (The primary fine-tuned model).
2. **HighRes_v2**: YOLOv10s (Small), imgsz=800, 5 epochs (Testing high-resolution detail).
3. **QuickTest_v3**: YOLOv10n (Nano), imgsz=640, 5 epochs (Testing a lightweight architecture).

In [ ]:
from ultralytics import YOLO
import os
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import pathlib

# 1. Find Project Root
cwd = os.getcwd()
if os.path.basename(cwd) == 'notebooks':
    ROOT_DIR = os.path.abspath(os.path.join(cwd, '..'))
else:
    ROOT_DIR = cwd

print(f"Project Root: {ROOT_DIR}")

# 2. Fix MLflow Windows Path Issue (KeyError: 'c')
mlflow_path = pathlib.Path(ROOT_DIR) / "mlruns"
tracking_uri = mlflow_path.as_uri()
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("Prometheus_Experimentation")
print(f"MLflow Tracking URI: {tracking_uri}")

# 3. Prepare Absolute Path YAML
dataset_path = os.path.join(ROOT_DIR, 'model/data/final_dataset')
if not os.path.exists(dataset_path):
    dataset_path = os.path.join(ROOT_DIR, 'model/data/test_frames')

data_config = {
    'path': dataset_path,
    'train': 'images/train',
    'val': 'images/val',
    'names': { 0: 'person', 1: 'gym-machine' }
}

temp_yaml_path = os.path.join(ROOT_DIR, 'model/data/prometheus_experiment.yml')
with open(temp_yaml_path, 'w') as f:
    yaml.dump(data_config, f)

# 4. Run QuickTest_v3 (if not already run or if crashed)
quick_test_results = os.path.join(ROOT_DIR, 'runs/detect/prometheus_runs/QuickTest_v3/results.csv')
if not os.path.exists(quick_test_results):
    print("Running QuickTest_v3 (Nano model)...")
    model_n = YOLO('yolov10n.pt')
    model_n.train(
        data=temp_yaml_path,
        epochs=5,
        imgsz=640,
        batch=16,
        device='cpu',
        project='prometheus_runs',
        name='QuickTest_v3',
        freeze=10,
        exist_ok=True
    )
else:
    print("✅ QuickTest_v3 results found.")

Project Root: c:\Users\Phasit\Desktop\Prometheus\Prometheus
MLflow Tracking URI: file:///c:/Users/Phasit/Desktop/Prometheus/Prometheus/mlruns
Running QuickTest_v3 (Nano model)...


c:\Users\Phasit\Desktop\Prometheus\Prometheus\venv\lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


New https://pypi.org/project/ultralytics/8.4.50 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.48  Python-3.10.6 torch-2.11.0+cpu CPU (13th Gen Intel Core i5-13500HX)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\Phasit\Desktop\Prometheus\Prometheus\model/data/prometheus_experiment.yml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10n.pt

## 2. Quantitative Comparison Table
We extract the final metrics (mAP@0.5) from the `results.csv` files of each run.

In [ ]:
def get_metrics(run_name):
    csv_path = os.path.join(ROOT_DIR, f'runs/detect/prometheus_runs/{run_name}/results.csv')
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()
        final_row = df.iloc[-1]
        return {
            'Run Name': run_name,
            'mAP@0.5': round(final_row['metrics/mAP50(B)'], 4),
            'Precision': round(final_row['metrics/precision(B)'], 4),
            'Recall': round(final_row['metrics/recall(B)'], 4),
            'Training Time (s)': round(df['time'].sum(), 2)
        }
    return None

runs = ['baseline_model', 'HighRes_v2', 'QuickTest_v3']
comparison_data = []

for run in runs:
    m = get_metrics(run)
    if m: comparison_data.append(m)

comparison_df = pd.DataFrame(comparison_data)
comparison_df.set_index('Run Name', inplace=True)
display(comparison_df)

,mAP@0.5,Precision,Recall,Training Time (s)
Run Name,,,,
baseline_model,0.9870,0.9711,0.9583,2296.47
HighRes_v2,0.1719,0.3797,0.1667,44.76
QuickTest_v3,0.8020,0.7951,0.6723,1551.14


## 3. Justification & Final Selection

### Analysis:
- **Baseline_v1 (YOLOv10s)**: Achieved the highest mAP. This architecture provides the best balance for occupancy detection where missing a person (False Negative) is the main operational risk.
- **HighRes_v2**: Testing at 800px image size. Useful for fine-grained detection but significantly increases inference latency.
- **QuickTest_v3 (YOLOv10n)**: The fastest architecture. Ideal for mobile devices but shows a drop in detection recall compared to the Small (s) version.

### Final Model Selection: **Baseline_v1**
We select **Baseline_v1** as our production candidate because it comfortably exceeds our performance target of **mAP@0.5 >= 0.85** and provides the most robust detections across all test scenarios.